# 🩸 Raktabeej — Video Reconstruction from Dataset

**What is this notebook?**
This notebook takes a dataset ZIP exported from [Raktabeej](https://raktabeej-5mbx.onrender.com)
and reconstructs a video from the extracted frames using OpenCV.

Raktabeej is a browser-based dataset generation engine that converts any video into
a structured, named image dataset — entirely in your browser, with zero server involvement.
This notebook is the final proof of the pipeline: the frames that were born from a video
can be reassembled back into one.

---

**What this notebook does — step by step**

| Step | Action |
|------|--------|
| 1 | Extracts the dataset ZIP into a `dataset/` folder |
| 2 | Reads `manifest.json` to get frames in the correct segment order |
| 3 | Detects the native frame resolution from the first image |
| 4 | Calculates FPS from `frames_per_segment ÷ segment_size_seconds` |
| 5 | Writes all frames sequentially into `reconstructed.mp4` using OpenCV |

---

**Expected input**
Upload a ZIP exported from Raktabeej — full dataset format:

raktabeej_dataset_yourfile.zip

├── frames/

│   ├── yourfile_segment_01_frame_001.jpg

│   ├── yourfile_segment_01_frame_002.jpg

│   └── ...

└── manifest.json

**Expected output**
`reconstructed.mp4` — a playable video rebuilt from the extracted frames.

---

**Known limitations**
- JPEG compression artifacts are present (frames were encoded at 85% quality)
- Audio is not preserved — Raktabeej extracts visual frames only
- If `frames_per_segment` is less than the original video FPS, the reconstruction
  plays slower than the source
- The result is visually correct but not bit-identical to the original

---

*Part of the Raktabeej pipeline — [github.com/ArunVijaykumarcsds/raktabeej](https://github.com/ArunVijaykumarcsds/raktabeej)*

In [1]:
import zipfile, json, cv2, numpy as np
from pathlib import Path

# 1. Extract ZIP
with zipfile.ZipFile('/content/raktabeej_dataset_test.zip', 'r') as z:
    z.extractall('dataset/')

# 2. Load manifest
with open('dataset/manifest.json') as f:
    manifest = json.load(f)

# 3. Get all frames in correct order
all_frames = []
for seg in manifest['segments']:
    all_frames.extend(seg['frames'])

# 4. Read first frame to get dimensions
first = cv2.imread(f"dataset/frames/{all_frames[0]}")
h, w = first.shape[:2]

# 5. Write video
fps = manifest['frames_per_segment'] / manifest['segment_size_seconds']  # = 25
out = cv2.VideoWriter('reconstructed.mp4',
                       cv2.VideoWriter_fourcc(*'mp4v'),
                       fps, (w, h))

for filename in all_frames:
    frame = cv2.imread(f"dataset/frames/{filename}")
    if frame is not None:
        out.write(frame)

out.release()
print(f"Done — {len(all_frames)} frames at {fps}fps")

Done — 250 frames at 25.0fps
